# 02 - Exploratory Data Analysis

Fraud distribution, amount profiles, hourly fraud rates and the new-vs-known
customer / device gaps that motivate the project.


In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.config import get_settings
from src.data_loader import load_dataframe, valid_numeric

cfg = get_settings()
df = valid_numeric(load_dataframe(cfg)[0])

f, axes = plt.subplots(2, 2, figsize=(12, 7))
df['is_fraud'].value_counts().sort_index().plot.bar(
    ax=axes[0,0], color=['#90caf9', '#e57373'])
axes[0,0].set_title("class balance")
df.groupby('transaction_hour')['is_fraud'].mean().plot(
    ax=axes[0,1], color='#5c6bc0')
axes[0,1].set_title("fraud rate by hour")
df.groupby('new_customer')['is_fraud'].mean().plot.bar(
    ax=axes[1,0], color=['#66bb6a', '#ef5350'])
axes[1,0].set_title("new vs known customers")
df.groupby('new_device')['is_fraud'].mean().plot.bar(
    ax=axes[1,1], color=['#66bb6a', '#ef5350'])
axes[1,1].set_title("new vs known devices")
plt.tight_layout(); plt.show()


In [ ]:
print("fraud rate, all             : {:.3f}%".format(df['is_fraud'].mean()*100))
print("fraud rate, new customers   : {:.3f}%".format(
    df.loc[df['new_customer']==1, 'is_fraud'].mean()*100))
print("fraud rate, known customers : {:.3f}%".format(
    df.loc[df['new_customer']==0, 'is_fraud'].mean()*100))
print("mean amount non-fraud: {:.2f} | fraud: {:.2f}".format(
    df.loc[df['is_fraud']==0, 'amount'].mean(),
    df.loc[df['is_fraud']==1, 'amount'].mean()))


## Observations
- Baseline fraud rate ~3%; new customers / devices carry a substantially
  higher rate – raw *novelty* cues are informative but would over-alert if
  used naively (that is why the risk engine requires a probability floor for
  the novelty->review channel).
